# NAME: LY CHUNGHEANG
# ID: e20220980
# GROUP: I5_AMS_A

# TP 3: Information Retrieval Models
## Boolean Retrieval and the Inverted Index



# Problem 1: Create a Term-Document Incidence Matrix

### 1.1 Sample documents (docIDs 1 to 6)

In [1]:
docs = {
    1: "Data science combines statistics, computer science, and domain knowledge.",
    2: "Machine learning algorithms can analyze large datasets and make predictions.",
    3: "Data visualization helps in interpreting complex data and communicating insights.",
    4: "Computer science students learn machine learning and data mining.",
    5: "Search engines use an inverted index to retrieve documents for a query.",
    6: "Statistics and machine learning help data scientists build predictive models.",
}
doc_ids = sorted(docs)
for d in doc_ids:
    print(f"Document {d}: {docs[d]}")

Document 1: Data science combines statistics, computer science, and domain knowledge.
Document 2: Machine learning algorithms can analyze large datasets and make predictions.
Document 3: Data visualization helps in interpreting complex data and communicating insights.
Document 4: Computer science students learn machine learning and data mining.
Document 5: Search engines use an inverted index to retrieve documents for a query.
Document 6: Statistics and machine learning help data scientists build predictive models.


### 1.2 Tokenizer and sorted vocabulary
Lowercase, remove punctuation, split into words.

In [2]:
import re

def tokenize(text):
    """Lowercase, strip punctuation, split into words."""
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    return text.split()

tokens = {d: tokenize(docs[d]) for d in doc_ids}
vocab = sorted({w for toks in tokens.values() for w in toks})

print("Tokens of Document 1:", tokens[1])
print(f"Vocabulary size: {len(vocab)}")
print(vocab)

Tokens of Document 1: ['data', 'science', 'combines', 'statistics', 'computer', 'science', 'and', 'domain', 'knowledge']
Vocabulary size: 44
['a', 'algorithms', 'an', 'analyze', 'and', 'build', 'can', 'combines', 'communicating', 'complex', 'computer', 'data', 'datasets', 'documents', 'domain', 'engines', 'for', 'help', 'helps', 'in', 'index', 'insights', 'interpreting', 'inverted', 'knowledge', 'large', 'learn', 'learning', 'machine', 'make', 'mining', 'models', 'predictions', 'predictive', 'query', 'retrieve', 'science', 'scientists', 'search', 'statistics', 'students', 'to', 'use', 'visualization']


### 1.3 Incidence matrix
Rows = terms, columns = documents; entry is 1 if the term occurs in the document, else 0.

In [3]:
import pandas as pd

matrix = {t: [1 if t in tokens[d] else 0 for d in doc_ids] for t in vocab}

df = pd.DataFrame.from_dict(matrix, orient="index", columns=[f"D{d}" for d in doc_ids])
df.index.name = "term"
pd.set_option("display.max_rows", None)
df

,D1,D2,D3,D4,D5,D6
term,,,,,,
a,0,0,0,0,1,0
algorithms,0,1,0,0,0,0
an,0,0,0,0,1,0
analyze,0,1,0,0,0,0
and,1,1,1,1,0,1
build,0,0,0,0,0,1
can,0,1,0,0,0,0
combines,1,0,0,0,0,0
communicating,0,0,1,0,0,0


### 1.4 Query: `data AND science AND NOT machine`
Complement of the `machine` row, then bitwise AND of the three vectors (as in Brutus AND Caesar AND NOT Calpurnia).

In [4]:
data, science, machine = matrix["data"], matrix["science"], matrix["machine"]
not_machine = [1 - b for b in machine]
result = [a & b & c for a, b, c in zip(data, science, not_machine)]

print("            " + "  ".join(f"D{d}" for d in doc_ids))
print("data        ", data)
print("science     ", science)
print("machine     ", machine)
print("NOT machine ", not_machine)
print("-" * 40)
print("RESULT      ", result)
print()
print("Matching documents:", [d for d, bit in zip(doc_ids, result) if bit])

            D1  D2  D3  D4  D5  D6
data         [1, 0, 1, 1, 0, 1]
science      [1, 0, 0, 1, 0, 0]
machine      [0, 1, 0, 1, 0, 1]
NOT machine  [1, 0, 1, 0, 1, 0]
----------------------------------------
RESULT       [1, 0, 0, 0, 0, 0]

Matching documents: [1]


**Therefore:** only **Document 1**. Document 4 has *data* and *science* but also *machine*, so it is excluded; Document 3 has *data* but not *science*. *scientists* (Doc 6) is a different term from *science* because no stemming is applied.


# Problem 2: Build an Inverted Index

### 2.1 (term, docID) pairs in document order

In [5]:
pairs = [(term, d) for d in doc_ids for term in tokenize(docs[d])]
print("Number of pairs:", len(pairs))
print(pairs[:12], "...")

Number of pairs: 60
[('data', 1), ('science', 1), ('combines', 1), ('statistics', 1), ('computer', 1), ('science', 1), ('and', 1), ('domain', 1), ('knowledge', 1), ('machine', 2), ('learning', 2), ('algorithms', 2)] ...


### 2.2 Sort by term, then by docID

In [6]:
sorted_pairs = sorted(pairs)          # tuples sort by term first, then docID
print(sorted_pairs[:15], "...")

[('a', 5), ('algorithms', 2), ('an', 5), ('analyze', 2), ('and', 1), ('and', 2), ('and', 3), ('and', 4), ('and', 6), ('build', 6), ('can', 2), ('combines', 1), ('communicating', 3), ('complex', 3), ('computer', 1)] ...


### 2.3 Merge duplicates and group into a dictionary
`term -> (document frequency, postings list)`. Duplicate (term, docID) pairs (e.g. *science* twice in Doc 1, *data* twice in Doc 3) are merged.

In [ ]:
index = {}
prev = None
for term, d in sorted_pairs:
    if (term, d) == prev:               # same term in same document: skip duplicate
        continue
    index.setdefault(term, []).append(d)
    prev = (term, d)

# final form: term -> (document frequency, postings list)
inverted_index = {t: (len(p), p) for t, p in index.items()}

print(f"{len(inverted_index)} terms in the dictionary")
print("science:", inverted_index["science"])

44 terms in the dictionary
science: (2, [1, 4])


### 2.4 Document frequency and postings lists of selected terms

In [ ]:
for t in ["data", "science", "machine", "learning", "statistics", "computer"]:
    dfreq, postings = inverted_index[t]
    print(f"{t:<11} df = {dfreq}   postings = {postings}")

data        df = 4   postings = [1, 3, 4, 6]
science     df = 2   postings = [1, 4]
machine     df = 3   postings = [2, 4, 6]
learning    df = 3   postings = [2, 4, 6]
statistics  df = 2   postings = [1, 6]
computer    df = 2   postings = [1, 4]


### 2.5 Incidence matrix vs. inverted index

In [ ]:
cells = len(vocab) * len(doc_ids)
postings_total = sum(df_ for df_, _ in inverted_index.values())
ones = sum(sum(row) for row in matrix.values())

print(f"Incidence matrix cells : {len(vocab)} x {len(doc_ids)} = {cells}")
print(f"Cells equal to 1       : {ones}")
print(f"Postings in the index  : {postings_total}")
print(f"Fraction of non-zero cells: {ones / cells:.1%}")

# Extrapolation to 1 million documents (typical orders of magnitude)
N, M, avg_unique_terms = 1_000_000, 500_000, 500
matrix_bits = N * M
postings_est = N * avg_unique_terms
print()
print(f"1M docs, 500k terms -> matrix: {matrix_bits:.1e} cells (~{matrix_bits/8/1e9:.0f} GB even at 1 bit/cell)")
print(f"1M docs, ~500 distinct terms/doc -> index: ~{postings_est:.1e} postings")
print(f"Matrix is about {matrix_bits / postings_est:,.0f}x larger than the number of postings")

Incidence matrix cells : 44 x 6 = 264
Cells equal to 1       : 58
Postings in the index  : 58
Fraction of non-zero cells: 22.0%

1M docs, 500k terms -> matrix: 5.0e+11 cells (~62 GB even at 1 bit/cell)
1M docs, ~500 distinct terms/doc -> index: ~5.0e+08 postings
Matrix is about 1,000x larger than the number of postings


**Discussion.** The two numbers are equal in meaning: the matrix stores one cell for every (term, document) pair, while the index stores only the pairs where the term actually occurs. The postings count equals the number of 1s in the matrix, and the rest of the matrix is zeros. On this toy collection about a quarter of the cells are 1, but that proportion falls sharply as the collection grows, because a document only contains a tiny fraction of the vocabulary.

For **1 million documents** I would use the **inverted index**:
- The matrix would have roughly 500 billion cells (about 62 GB at one bit per cell), and more than 99.9% would be zeros, whereas the index stores only about 500 million postings.
- Postings lists are sorted by docID, so AND, OR and AND NOT queries are linear merges of short lists, and compression (gap encoding) makes them smaller still.
- The matrix would force a scan over bit vectors of length 1 million for every term in a query.


# Problem 3: Merge Algorithm

### 3.1 INTERSECT (x AND y)
Two pointers walk the sorted lists together in O(x + y) time.

In [ ]:
def intersect(p1, p2):
    answer = []
    i = j = 0
    while i < len(p1) and j < len(p2):
        if p1[i] == p2[j]:
            answer.append(p1[i])
            i += 1
            j += 1
        elif p1[i] < p2[j]:
            i += 1
        else:
            j += 1
    return answer

### 3.2 OR and AND NOT as linear merges (no Python sets)

In [ ]:
def union(p1, p2):
    """x OR y"""
    answer = []
    i = j = 0
    while i < len(p1) and j < len(p2):
        if p1[i] == p2[j]:
            answer.append(p1[i]); i += 1; j += 1
        elif p1[i] < p2[j]:
            answer.append(p1[i]); i += 1
        else:
            answer.append(p2[j]); j += 1
    answer.extend(p1[i:])      # at most one of these two has leftovers
    answer.extend(p2[j:])
    return answer

def and_not(p1, p2):
    """x AND NOT y"""
    answer = []
    i = j = 0
    while i < len(p1) and j < len(p2):
        if p1[i] == p2[j]:
            i += 1; j += 1               # in both: drop it
        elif p1[i] < p2[j]:
            answer.append(p1[i]); i += 1  # not in p2: keep
        else:
            j += 1
    answer.extend(p1[i:])                # p2 exhausted: everything left is kept
    return answer

### 3.3 Queries

In [ ]:
def postings(term):
    return inverted_index[term][1] if term in inverted_index else []

q1 = intersect(postings("data"), postings("machine"))
q2 = union(postings("statistics"), postings("learning"))
q3 = and_not(postings("data"), postings("science"))

print("Query 1: data AND machine       ", postings("data"), "&", postings("machine"), "->", q1)
print("Query 2: statistics OR learning ", postings("statistics"), "|", postings("learning"), "->", q2)
print("Query 3: data AND NOT science   ", postings("data"), "-", postings("science"), "->", q3)

Query 1: data AND machine        [1, 3, 4, 6] & [2, 4, 6] -> [4, 6]
Query 2: statistics OR learning  [1, 6] | [2, 4, 6] -> [1, 2, 4, 6]
Query 3: data AND NOT science    [1, 3, 4, 6] - [1, 4] -> [3, 6]


**Check against the documents:** Doc 4 and Doc 6 contain both *data* and *machine*; *statistics* or *learning* appears in Docs 1, 2, 4, 6; *data* without *science* gives Docs 3 and 6.


# Problem 4: Query Optimization

### 4.1 Conjunctive query with comparison counting
Each iteration of the merge loop compares two docIDs, so it counts as one comparison. Terms are processed from the shortest postings list (smallest document frequency) to the longest.

In [ ]:
def intersect_count(p1, p2):
    """INTERSECT that also returns the number of docID comparisons."""
    answer, comps = [], 0
    i = j = 0
    while i < len(p1) and j < len(p2):
        comps += 1
        if p1[i] == p2[j]:
            answer.append(p1[i]); i += 1; j += 1
        elif p1[i] < p2[j]:
            i += 1
        else:
            j += 1
    return answer, comps

def conjunctive_query(terms, optimize=True):
    """t1 AND t2 AND ... AND tn. If optimize, merge in increasing order of document frequency."""
    if optimize:
        terms = sorted(terms, key=lambda t: inverted_index[t][0])
    result, total = postings(terms[0]), 0
    for t in terms[1:]:
        result, c = intersect_count(result, postings(t))
        total += c
        if not result:          # early exit: empty result stays empty
            break
    return terms, result, total

### 4.2 `data AND machine AND science`: given order vs. optimized order

In [ ]:
query = ["data", "machine", "science"]
for t in query:
    print(f"df({t}) = {inverted_index[t][0]}")
print()
for label, opt in [("Given order    ", False), ("Optimized order", True)]:
    order, res, comps = conjunctive_query(query, optimize=opt)
    print(f"{label}: {' AND '.join(order):<28} result = {res}   comparisons = {comps}")

df(data) = 4
df(machine) = 3
df(science) = 2

Given order    : data AND machine AND science result = [4]   comparisons = 7
Optimized order: science AND machine AND data result = [4]   comparisons = 6


The optimized order starts with the shortest list (*science*), so the first intermediate result is as small as possible and every later merge is against a short list. With only six documents the saving is small, but on a real collection it can be orders of magnitude.

### 4.3 `(statistics OR learning) AND (data OR computer) AND (science OR models)`
Estimate the size of each OR group by the sum of the document frequencies of its terms, then process the groups in increasing order of estimate.

In [ ]:
groups = [["statistics", "learning"], ["data", "computer"], ["science", "models"]]

def or_group(terms):
    res = postings(terms[0])
    for t in terms[1:]:
        res = union(res, postings(t))
    return res

estimates = [sum(inverted_index[t][0] for t in g) for g in groups]
for g, e in zip(groups, estimates):
    print(f"({' OR '.join(g)})  estimate = {' + '.join(str(inverted_index[t][0]) for t in g)} = {e}")

order = sorted(range(len(groups)), key=lambda k: estimates[k])
print("\nProcessing order:", [f"({' OR '.join(groups[k])})" for k in order])

result, total = None, 0
for k in order:
    lst = or_group(groups[k])
    print(f"  ({' OR '.join(groups[k])}) -> {lst}")
    if result is None:
        result = lst
    else:
        result, c = intersect_count(result, lst)
        total += c
print("\nFinal result:", result, f"  (comparisons in the AND merges: {total})")

(statistics OR learning)  estimate = 2 + 3 = 5
(data OR computer)  estimate = 4 + 2 = 6
(science OR models)  estimate = 2 + 1 = 3

Processing order: ['(science OR models)', '(statistics OR learning)', '(data OR computer)']
  (science OR models) -> [1, 4, 6]
  (statistics OR learning) -> [1, 2, 4, 6]
  (data OR computer) -> [1, 3, 4, 6]

Final result: [1, 4, 6]   (comparisons in the AND merges: 8)



# Problem 5: Phrase Queries with a Positional Index

### 5.1 Positional index
`term -> {docID: [positions]}`, positions counted from 0 in the token list of each document.

In [ ]:
from collections import defaultdict

positional_index = defaultdict(dict)
for d in doc_ids:
    for pos, term in enumerate(tokens[d]):
        positional_index[term].setdefault(d, []).append(pos)
positional_index = dict(sorted(positional_index.items()))

for t in ["data", "science", "machine", "learning", "computer"]:
    print(f"{t:<9}", positional_index[t])

data      {1: [0], 3: [0, 6], 4: [7], 6: [5]}
science   {1: [1, 5], 4: [1]}
machine   {2: [0], 4: [4], 6: [2]}
learning  {2: [1], 4: [5], 6: [3]}
computer  {1: [4], 4: [0]}


### 5.2 Phrase query
First find the documents that contain all terms (using the linear `intersect`), then check that the terms appear at consecutive positions p, p+1, ..., p+n-1.

In [ ]:
def phrase_query(phrase):
    terms = tokenize(phrase)
    if any(t not in positional_index for t in terms):
        return []
    # candidate documents: those containing every term
    candidates = sorted(positional_index[terms[0]])
    for t in terms[1:]:
        candidates = intersect(candidates, sorted(positional_index[t]))
    # positional check
    matches = []
    for d in candidates:
        for p in positional_index[terms[0]][d]:
            if all((p + k) in positional_index[terms[k]][d] for k in range(1, len(terms))):
                matches.append(d)
                break
    return matches

### 5.3 Phrase queries

In [ ]:
for q in ["machine learning", "computer science", "data science"]:
    print(f'"{q}" -> {phrase_query(q)}')

"machine learning" -> [2, 4, 6]
"computer science" -> [1, 4]
"data science" -> [1]


### 5.4 `"data science"` vs. `data AND science`

In [ ]:
phrase_res = phrase_query("data science")
bool_res = intersect(postings("data"), postings("science"))
print('Phrase "data science" :', phrase_res)
print("Boolean data AND science:", bool_res)
print("Returned by Boolean only:", [d for d in bool_res if d not in phrase_res])
print()
print("Doc 4 positions -> data:", positional_index["data"][4], " science:", positional_index["science"][4])
print("Doc 4 tokens    :", list(enumerate(tokens[4])))

Phrase "data science" : [1]
Boolean data AND science: [1, 4]
Returned by Boolean only: [4]

Doc 4 positions -> data: [7]  science: [1]
Doc 4 tokens    : [(0, 'computer'), (1, 'science'), (2, 'students'), (3, 'learn'), (4, 'machine'), (5, 'learning'), (6, 'and'), (7, 'data'), (8, 'mining')]


**Answer:** **Document 4** is returned by the Boolean query `data AND science` but not by the phrase query `"data science"`.

Document 4 ("Computer science students learn machine learning and data mining") contains both words, so the Boolean model, which only checks presence, accepts it. But *science* is at position 1 (in "computer science") and *data* is at position 7 (in "data mining"), so they are never adjacent and *data* does not directly precede *science*. Only Document 1 contains the actual phrase "Data science". The positional index therefore gives higher precision than the plain inverted index, at the cost of a larger index.